# T03: Building a Custom Schema from a Python Dict

## What You'll Learn

In this tutorial you will build a data-generation schema entirely from a Python dictionary.  
This is the most flexible way to use Spindle: you define tables, columns, relationships,
and generation strategies in plain Python — no YAML, no external files.

We will cover **one input method** in depth — the Python dict — and along the way
demonstrate four powerful generation strategies:

| Strategy | Purpose |
|---|---|
| `weighted_enum` | Pick from a list with custom probabilities |
| `distribution` | Draw numeric values from a statistical distribution |
| `foreign_key` (Pareto) | Assign child rows to parents following the 80/20 rule |
| `formula` | Compute a column from other columns |

## Prerequisites

- `sqllocks-spindle` installed (`pip install sqllocks-spindle`)
- Completed **T01** and **T02**, or comfortable with `Spindle.generate()`

## Time Estimate

**~15 minutes**

## Step 1 — Import Spindle

**What we're about to do:** Import the core `Spindle` class so we can feed it our custom schema.

**Why this matters:** Every Spindle workflow starts with this single import. Nothing else is
needed for dict-based schemas because the dict itself carries all configuration.

**What to expect:** A simple confirmation print — no errors means Spindle is ready.

In [1]:
from sqllocks_spindle import Spindle

print("Spindle imported successfully.")

Spindle imported successfully.


## Step 2 — Define a Two-Table Schema as a Python Dict

**What we're about to do:** Build a complete schema dict describing a tiny company model
with two tables — `department` and `employee` — linked by a foreign key.

**Why this matters:** The dict is the canonical input format for Spindle.  Understanding its
structure unlocks every advanced feature: custom generators, distributions, formulas, and
business rules.  The top-level keys are:

- **`model`** — metadata (name, domain, locale, seed, date range)
- **`tables`** — one entry per table; each entry lists its columns and their generators
- **`relationships`** — foreign-key links between tables
- **`business_rules`** — cross-column constraints (empty for now)
- **`generation`** — row-count scales

Each column's `generator` field is a **dict** with a `strategy` key (e.g.,
`{"strategy": "sequence"}`, `{"strategy": "weighted_enum", "values": {"a": 0.5, "b": 0.5}}`)
plus any strategy-specific parameters.

**What to expect:** A printed summary showing two tables and their row counts.

In [2]:
schema = {
    "model": {
        "name": "my_company",
        "domain": "custom",
        "schema_mode": "3nf",
        "locale": "en_US",
        "seed": 42,
        "date_range": {"start": "2023-01-01", "end": "2025-12-31"},
    },
    "tables": {
        "department": {
            "columns": {
                "department_id": {"type": "integer", "generator": {"strategy": "sequence"}},
                "name": {
                    "type": "string",
                    "generator": {
                        "strategy": "weighted_enum",
                        "values": {
                            "Engineering": 1, "Sales": 1, "Marketing": 1,
                            "Finance": 1, "HR": 1, "Legal": 1,
                            "Operations": 1, "Support": 1, "Product": 1, "Design": 1,
                        },
                    },
                },
            },
            "primary_key": ["department_id"],
        },
        "employee": {
            "columns": {
                "employee_id": {"type": "integer", "generator": {"strategy": "sequence"}},
                "first_name": {"type": "string", "generator": {"strategy": "faker", "provider": "first_name"}},
                "last_name": {"type": "string", "generator": {"strategy": "faker", "provider": "last_name"}},
                "department_id": {"type": "integer", "generator": {"strategy": "foreign_key", "ref": "department.department_id"}},
                "status": {
                    "type": "string",
                    "generator": {
                        "strategy": "weighted_enum",
                        "values": {"active": 0.80, "on_leave": 0.10, "terminated": 0.10},
                    },
                },
                "monthly_salary": {
                    "type": "float",
                    "generator": {
                        "strategy": "distribution",
                        "distribution": "log_normal",
                        "mean": 8.5,
                        "sigma": 0.4,
                    },
                },
                "hire_date": {"type": "date", "generator": {"strategy": "temporal"}},
            },
            "primary_key": ["employee_id"],
        },
    },
    "relationships": [
        {
            "name": "employee_department",
            "parent": "department",
            "child": "employee",
            "parent_columns": ["department_id"],
            "child_columns": ["department_id"],
            "type": "many_to_one",
            "cardinality": {"distribution": "pareto"},
        }
    ],
    "business_rules": [],
    "generation": {
        "scales": {
            "small": {"department": 10, "employee": 200},
        }
    },
}

spindle = Spindle()
data = spindle.generate(schema=schema, scale="small")

print(f"Tables generated: {data.table_names}")
for name, df in data.tables.items():
    print(f"  {name}: {len(df)} rows, {list(df.columns)}")

Tables generated: ['department', 'employee']
  department: 10 rows, ['department_id', 'name']
  employee: 200 rows, ['employee_id', 'department_id', 'first_name', 'last_name', 'status', 'monthly_salary', 'hire_date']


## Step 3 — Verify Table Names and Foreign-Key Integrity

**What we're about to do:** Print both table names and confirm that every `department_id`
in the `employee` table actually exists in the `department` table.

**Why this matters:** Referential integrity is the whole point of declaring relationships.
If Spindle generated orphan rows, any downstream JOIN would silently lose data.
This quick check proves the FK contract is honored.

**What to expect:** A list of department IDs, then a boolean check confirming
zero orphan employee rows.

In [3]:
dept_ids = set(data["department"]["department_id"])
emp_dept_ids = set(data["employee"]["department_id"])

print("Department IDs:", sorted(dept_ids))
print(f"Employee references {len(emp_dept_ids)} unique departments.")

orphans = emp_dept_ids - dept_ids
print(f"Orphan department_ids in employee table: {orphans if orphans else 'None — FK integrity confirmed!'})")

Department IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Employee references 10 unique departments.
Orphan department_ids in employee table: None — FK integrity confirmed!)


## Step 4 — Weighted Enum: Employee Status

**What we're about to do:** Inspect the `status` column of the `employee` table to see
how the `weighted_enum` strategy distributed values.

**Why this matters:** Real-world categorical data is almost never uniformly distributed.
In any company most employees are active, a small fraction are on leave, and another small
fraction have been terminated.  The `weighted_enum` strategy lets you encode these
real-world proportions directly as a dict mapping values to weights:

```python
"values": {"active": 0.80, "on_leave": 0.10, "terminated": 0.10}
```

Spindle draws each row's status by weighted random selection, so the resulting
distribution will approximate (but not exactly match) the requested weights.

**What to expect:** A `value_counts()` table showing ~80% active, ~10% on_leave,
~10% terminated.

In [4]:
status_counts = data["employee"]["status"].value_counts()
status_pct = data["employee"]["status"].value_counts(normalize=True).mul(100).round(1)

print("Employee Status Distribution")
print("=" * 35)
for val in status_counts.index:
    print(f"  {val:<15} {status_counts[val]:>4} rows  ({status_pct[val]}%)")
print(f"\nTotal employees: {len(data['employee'])}")

Employee Status Distribution


  active           159 rows  (79.5%)
  on_leave          23 rows  (11.5%)
  terminated        18 rows  (9.0%)

Total employees: 200


## Step 5 — Distribution Strategy: Monthly Salary

**What we're about to do:** Print summary statistics for the `monthly_salary` column,
which was generated using a **log-normal distribution**.

**Why this matters:** Salary data in the real world is right-skewed: most people earn
a moderate amount, but a long tail of high earners stretches the distribution upward.
A log-normal distribution captures this shape perfectly.  We configured:

- `mean = 8.5` (the mean of the underlying *log* values)
- `sigma = 0.4` (the standard deviation of the log values)

Because `exp(8.5) ≈ 4,915`, the median salary should land around \$4,900 per month
(\~\$59k/year), with a right tail reaching well above that.

**What to expect:** Descriptive stats (mean, std, min, median, max) confirming the
right-skewed shape.

In [5]:
salary = data["employee"]["monthly_salary"]

print("Monthly Salary Distribution")
print("=" * 35)
print(f"  Mean:   ${salary.mean():>10,.2f}")
print(f"  Std:    ${salary.std():>10,.2f}")
print(f"  Min:    ${salary.min():>10,.2f}")
print(f"  Median: ${salary.median():>10,.2f}")
print(f"  Max:    ${salary.max():>10,.2f}")
print(f"\nSkewness: {salary.skew():.2f}  (positive = right-skewed, as expected)")

Monthly Salary Distribution
  Mean:   $  5,390.49
  Std:    $  2,136.29
  Min:    $  1,932.47
  Median: $  5,115.38
  Max:    $ 13,892.01

Skewness: 0.90  (positive = right-skewed, as expected)


## Step 6 — Foreign Key with Pareto Distribution

**What we're about to do:** Count how many employees belong to each department and
observe the Pareto (80/20) effect.

**Why this matters:** When we defined the `employee → department` relationship we set
`"distribution": "pareto"`.  This tells Spindle to assign employees to departments
following the 80/20 rule: a small number of departments get the lion's share of employees,
while most departments remain relatively small.

This is realistic — in most organizations, Engineering and Sales dwarf departments
like Legal or Design.

**What to expect:** A sorted list of department sizes.  The top 2–3 departments should
contain a disproportionately large share of the 200 employees.

In [6]:
import pandas as pd

dept_counts = (
    data["employee"]
    .merge(data["department"], on="department_id")
    .groupby("name")
    .size()
    .sort_values(ascending=False)
)

print("Employees per Department (Pareto FK distribution)")
print("=" * 50)
cumulative = 0
for dept, count in dept_counts.items():
    cumulative += count
    pct = count / len(data["employee"]) * 100
    cum_pct = cumulative / len(data["employee"]) * 100
    print(f"  {dept:<15} {count:>4} employees  ({pct:5.1f}%)  cumulative: {cum_pct:5.1f}%")

Employees per Department (Pareto FK distribution)
  Support           66 employees  ( 33.0%)  cumulative:  33.0%
  HR                35 employees  ( 17.5%)  cumulative:  50.5%
  Engineering       30 employees  ( 15.0%)  cumulative:  65.5%
  Operations        19 employees  (  9.5%)  cumulative:  75.0%
  Product           17 employees  (  8.5%)  cumulative:  83.5%
  Sales             17 employees  (  8.5%)  cumulative:  92.0%
  Design            16 employees  (  8.0%)  cumulative: 100.0%


## Step 7 — Formula Strategy: Computed Columns

**What we're about to do:** Add a computed column — `annual_salary` — that is derived
from `monthly_salary * 12` using the `formula` generator strategy.

**Why this matters:** Many analytics pipelines expect pre-computed columns (annual salary,
full name, age buckets, etc.).  Rather than adding a post-processing step after generation,
the `formula` strategy lets you declare these derivations *inside the schema itself*.
This keeps your schema self-documenting and ensures the computed column is always present.

We will modify our schema to include the new column, regenerate, and verify that
`annual_salary == monthly_salary * 12` for every row.

**What to expect:** A sample of rows showing both `monthly_salary` and `annual_salary`,
plus a boolean check confirming the formula holds for all 200 employees.

In [7]:
# Add a formula column to the employee table
schema["tables"]["employee"]["columns"]["annual_salary"] = {
    "type": "float",
    "generator": {
        "strategy": "formula",
        "expression": "monthly_salary * 12",
    },
}

# Regenerate with the updated schema
spindle = Spindle()
data = spindle.generate(schema=schema, scale="small")

emp = data["employee"]
print("Sample: monthly vs. annual salary")
print(emp[["first_name", "last_name", "monthly_salary", "annual_salary"]].head(8).to_string(index=False))

# Verify the formula
match = (emp["annual_salary"].round(2) == (emp["monthly_salary"] * 12).round(2)).all()
print(f"\nFormula check (annual == monthly * 12): {match}")

Sample: monthly vs. annual salary
first_name last_name  monthly_salary  annual_salary
    Teresa Rodriguez     5603.788655   67245.463859
      Anne   Edwards     9807.866882  117694.402581
     Jamie  Woodward     2660.995123   31931.941470
     Terry    Daniel     6943.291395   83319.496740
      Sean    Jordan     4309.554919   51714.659030
     James Alexander     4795.677458   57548.129500
    Thomas    Wilson     3225.491419   38705.897032
      Juan      Wade     4299.343136   51592.117638

Formula check (annual == monthly * 12): True


## What's Next

You now know how to build a full schema from a Python dict and leverage four key
generation strategies: `weighted_enum`, `distribution`, `foreign_key` (Pareto), and `formula`.

### Recommended next steps

| Notebook / Resource | What You'll Learn |
|---|---|
| **T05 — Distribution Overrides** | Fine-tune any column's statistical distribution after the fact |
| **Strategies Reference** (`strategies.md`) | Full catalog of every generator strategy with examples |

### Quick links

- [T05_distribution_overrides.ipynb](./T05_distribution_overrides.ipynb)
- [Strategies Reference](../../reference/strategies.md)